In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

Initial Load

In [0]:
src_customers_df = spark.read.csv(path="/Volumes/workspace/mydb/myvolume/customer_src/customers.csv", header=True, inferSchema=True)

src_customers_df.show()

+-----------+------+------+------+
|customer_id|  name|  city|salary|
+-----------+------+------+------+
|          1| Ayush| Noida| 30000|
|          2|Piyush|Sorkha| 20000|
|          3|Gaurav|Colony| 34000|
|          4| Bhonu|Sorkha| 20000|
+-----------+------+------+------+



In [0]:
src_customers_df = src_customers_df.withColumn("is_active", lit('Y')).withColumn("start_date", current_date()).withColumn("end_date", lit("9999-12-31").cast("date"))

src_customers_df.show()

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-6068792838066430>, line 1
----> 1 src_customers_df = src_customers_df.withColumn("is_active", lit('Y')).withColumn("start_date", current_date()).withColumn("end_date", lit("9999-12-31").cast("date"))
      3 src_customers_df.show()

NameError: name 'src_customers_df' is not defined

In [0]:
src_customers_df.write.mode("append").saveAsTable("mydb.CUST_DIM")

In [0]:
%sql
select * from mydb.cust_dim;

customer_id,name,city,salary,is_active,start_date,end_date
1,Ayush,Noida,30000,Y,2026-07-19,9999-12-31
2,Piyush,Sorkha,20000,Y,2026-07-19,9999-12-31
3,Gaurav,Colony,34000,Y,2026-07-19,9999-12-31
4,Bhonu,Sorkha,20000,Y,2026-07-19,9999-12-31


## SCD Type 2

In [0]:
# STEP 1: Read the data from the source
src_customers_df = spark.read.csv(path="/Volumes/workspace/mydb/myvolume/customer_src/customers.csv", header=True, inferSchema=True)

src_customers_df.show()

+-----------+--------+--------+------+
|customer_id|    name|    city|salary|
+-----------+--------+--------+------+
|          1|   Ayush|   Noida| 50000|
|          3|  Gourav|  Colony| 34000|
|          4|   Bhonu|  Sorkha| 20000|
|          5|Pragnesh|Gr.Noida| 33000|
+-----------+--------+--------+------+



In [0]:
# STEP 2: Read from target or dimensions table and filter the active records
tgt_customers_df = spark.sql("select * from mydb.cust_dim where is_active = 'Y'")

tgt_customers_df.show()

+-----------+------+------+------+---------+----------+----------+
|customer_id|  name|  city|salary|is_active|start_date|  end_date|
+-----------+------+------+------+---------+----------+----------+
|          1| Ayush| Noida| 30000|        Y|2026-07-19|9999-12-31|
|          2|Piyush|Sorkha| 20000|        Y|2026-07-19|9999-12-31|
|          3|Gaurav|Colony| 34000|        Y|2026-07-19|9999-12-31|
|          4| Bhonu|Sorkha| 20000|        Y|2026-07-19|9999-12-31|
+-----------+------+------+------+---------+----------+----------+



In [0]:
# STEP 3: Adding hash key to both the dfs

src_customers_df = src_customers_df.withColumn("src_hash_key", xxhash64("name", "city", "salary"))
src_customers_df.show()

tgt_customers_df = tgt_customers_df.withColumn("tgt_hash_key", xxhash64("name", "city", "salary"))
tgt_customers_df.show()

+-----------+--------+--------+------+--------------------+
|customer_id|    name|    city|salary|        src_hash_key|
+-----------+--------+--------+------+--------------------+
|          1|   Ayush|   Noida| 50000|-2909879855720589612|
|          3|  Gourav|  Colony| 34000|-4066889726975237581|
|          4|   Bhonu|  Sorkha| 20000| 8319510517518466142|
|          5|Pragnesh|Gr.Noida| 33000| -647272837507553151|
+-----------+--------+--------+------+--------------------+

+-----------+------+------+------+---------+----------+----------+--------------------+
|customer_id|  name|  city|salary|is_active|start_date|  end_date|        tgt_hash_key|
+-----------+------+------+------+---------+----------+----------+--------------------+
|          1| Ayush| Noida| 30000|        Y|2026-07-19|9999-12-31|-5123315276963748403|
|          2|Piyush|Sorkha| 20000|        Y|2026-07-19|9999-12-31| -818599330964549460|
|          3|Gaurav|Colony| 34000|        Y|2026-07-19|9999-12-31| -76531013057

In [0]:
# STEP 4: Rename source and target columns to match the target table

src_customers_df = src_customers_df.withColumnRenamed("customer_id", "src_customer_id").withColumnRenamed("name", "src_name").withColumnRenamed("city", "src_city").withColumnRenamed("salary", "src_salary")
src_customers_df.show()

tgt_customers_df = tgt_customers_df.withColumnRenamed("customer_id", "tgt_customer_id").withColumnRenamed("name", "tgt_name").withColumnRenamed("city", "tgt_city").withColumnRenamed("salary", "tgt_salary")
tgt_customers_df.show()

+---------------+--------+--------+----------+--------------------+
|src_customer_id|src_name|src_city|src_salary|        src_hash_key|
+---------------+--------+--------+----------+--------------------+
|              1|   Ayush|   Noida|     50000|-2909879855720589612|
|              3|  Gourav|  Colony|     34000|-4066889726975237581|
|              4|   Bhonu|  Sorkha|     20000| 8319510517518466142|
|              5|Pragnesh|Gr.Noida|     33000| -647272837507553151|
+---------------+--------+--------+----------+--------------------+

+---------------+--------+--------+----------+---------+----------+----------+--------------------+
|tgt_customer_id|tgt_name|tgt_city|tgt_salary|is_active|start_date|  end_date|        tgt_hash_key|
+---------------+--------+--------+----------+---------+----------+----------+--------------------+
|              1|   Ayush|   Noida|     30000|        Y|2026-07-19|9999-12-31|-5123315276963748403|
|              2|  Piyush|  Sorkha|     20000|        Y

In [0]:
# STEP 5: Join the source and target

joined_df = src_customers_df.alias("src").join(other=tgt_customers_df.alias("tgt"), on=src_customers_df.src_customer_id == tgt_customers_df.tgt_customer_id, how="left")

joined_df.display()

src_customer_id,src_name,src_city,src_salary,src_hash_key,tgt_customer_id,tgt_name,tgt_city,tgt_salary,is_active,start_date,end_date,tgt_hash_key
1,Ayush,Noida,50000,-2909879855720589612,1,Ayush,Noida,30000,Y,2026-07-19,9999-12-31,-5123315276963748403
3,Gourav,Colony,34000,-4066889726975237581,3,Gaurav,Colony,34000,Y,2026-07-19,9999-12-31,-765310130576577486
4,Bhonu,Sorkha,20000,8319510517518466142,4,Bhonu,Sorkha,20000,Y,2026-07-19,9999-12-31,8319510517518466142
5,Pragnesh,Gr.Noida,33000,-647272837507553151,null,null,null,null,null,null,null,null


In [0]:
# STEP 6: Find chaneged records (already exists in target but updated in source)
changed_recrods_df = joined_df.filter("tgt_customer_id IS NOT NULL AND src_hash_key != tgt_hash_key")

changed_recrods_df.display()

src_customer_id,src_name,src_city,src_salary,src_hash_key,tgt_customer_id,tgt_name,tgt_city,tgt_salary,is_active,start_date,end_date,tgt_hash_key
1,Ayush,Noida,50000,-2909879855720589612,1,Ayush,Noida,30000,Y,2026-07-19,9999-12-31,-5123315276963748403
3,Gourav,Colony,34000,-4066889726975237581,3,Gaurav,Colony,34000,Y,2026-07-19,9999-12-31,-765310130576577486


In [0]:
# STEP 6a: Create new version of changed records
new_version_records_df = (
changed_recrods_df.select("src.*")
.withColumn("is_active", lit('Y')).withColumn("start_date", current_date()).withColumn("end_date", lit("9999-12-31").cast("date"))
.withColumnsRenamed({"src_customer_id": "customer_id", "src_name": "name", "src_city": "city", "src_salary": "salary"})
.drop("src_hash_key")
)

new_version_records_df.display()

customer_id,name,city,salary,is_active,start_date,end_date
1,Ayush,Noida,50000,Y,2026-07-19,9999-12-31
3,Gourav,Colony,34000,Y,2026-07-19,9999-12-31


In [0]:
# STEP 6b: End the existing records in target
ended_old_records_df = (
changed_recrods_df.select("tgt.*")
.withColumn("is_active", lit('N')).withColumn("end_date", current_date() - 1)
.withColumnsRenamed({"tgt_customer_id": "customer_id", "tgt_name": "name", "tgt_city": "city", "tgt_salary": "salary"})
.drop("tgt_hash_key")
)

ended_old_records_df.display()

customer_id,name,city,salary,is_active,start_date,end_date
1,Ayush,Noida,30000,N,2026-07-19,2026-07-18
3,Gaurav,Colony,34000,N,2026-07-19,2026-07-18


In [0]:
# STEP 7: Finding new added records
new_records = joined_df.filter("tgt_customer_id IS NULL")

new_records = (
new_records.select("src.*")
.withColumn("is_active", lit('Y')).withColumn("start_date", current_date()).withColumn("end_date", lit("9999-12-31").cast("date"))
.withColumnsRenamed({"src_customer_id": "customer_id", "src_name": "name", "src_city": "city", "src_salary": "salary"})
.drop("src_hash_key")
)

new_records.display()

customer_id,name,city,salary,is_active,start_date,end_date
5,Pragnesh,Gr.Noida,33000,Y,2026-07-19,9999-12-31


In [0]:
# STEP 8: Finding unchanged records from source
unchanged_records_df = joined_df.filter("tgt_customer_id IS NOT NULL AND src_hash_key = tgt_hash_key")

unchanged_records_df = (
unchanged_records_df.select("tgt.*")
.withColumnsRenamed({"tgt_customer_id": "customer_id", "tgt_name": "name", "tgt_city": "city", "tgt_salary": "salary"})
.drop("tgt_hash_key")
)

unchanged_records_df.display()


customer_id,name,city,salary,is_active,start_date,end_date
4,Bhonu,Sorkha,20000,Y,2026-07-19,9999-12-31


In [0]:
# STEP 9: Finding missing records (present in target but not in source)
missing_records = tgt_customers_df.join(src_customers_df, on=tgt_customers_df.tgt_customer_id == src_customers_df.src_customer_id, how="leftanti")

missing_records = (
missing_records
.withColumnsRenamed({"tgt_customer_id": "customer_id", "tgt_name": "name", "tgt_city": "city", "tgt_salary": "salary"})
.drop("tgt_hash_key")
)

missing_records.display()

customer_id,name,city,salary,is_active,start_date,end_date
2,Piyush,Sorkha,20000,Y,2026-07-19,9999-12-31


In [0]:
# STEP 10: UNION ALL
tgt_customers_df = (
new_version_records_df
.unionAll(ended_old_records_df)
.unionAll(new_records)
.unionAll(unchanged_records_df)
.unionAll(missing_records)
)

tgt_customers_df.display()

tgt_customers_df.write.mode("overwrite").saveAsTable("mydb.CUST_DIM")

customer_id,name,city,salary,is_active,start_date,end_date
1,Ayush,Noida,50000,Y,2026-07-19,9999-12-31
3,Gourav,Colony,34000,Y,2026-07-19,9999-12-31
1,Ayush,Noida,30000,N,2026-07-19,2026-07-18
3,Gaurav,Colony,34000,N,2026-07-19,2026-07-18
5,Pragnesh,Gr.Noida,33000,Y,2026-07-19,9999-12-31
4,Bhonu,Sorkha,20000,Y,2026-07-19,9999-12-31
2,Piyush,Sorkha,20000,Y,2026-07-19,9999-12-31


## SCD Type 2 (with MERGE)

In [0]:
# STEP 1: Read the data from the source
src_customers_df = spark.read.csv(path="/Volumes/workspace/mydb/myvolume/customer_src/customers.csv", header=True, inferSchema=True)

src_customers_df.show()

+-----------+---------+--------+------+
|customer_id|     name|    city|salary|
+-----------+---------+--------+------+
|          1|    Ayush|   Noida| 50000|
|          5| Pragnesh|Gr.Noida| 34000|
|          6|Priyanshu|   Khora| 34000|
+-----------+---------+--------+------+



In [0]:
# STEP 2: Read from target or dimensions table and filter the active records
tgt_customers_df = spark.sql("select * from mydb.cust_dim where is_active = 'Y'")

tgt_customers_df.show()

+-----------+--------+--------+------+---------+----------+----------+
|customer_id|    name|    city|salary|is_active|start_date|  end_date|
+-----------+--------+--------+------+---------+----------+----------+
|          1|   Ayush|   Noida| 50000|        Y|2026-07-19|9999-12-31|
|          3|  Gourav|  Colony| 34000|        Y|2026-07-19|9999-12-31|
|          5|Pragnesh|Gr.Noida| 33000|        Y|2026-07-19|9999-12-31|
|          2|  Piyush|  Sorkha| 20000|        Y|2026-07-19|9999-12-31|
|          4|   Bhonu|  Sorkha| 20000|        Y|2026-07-19|9999-12-31|
+-----------+--------+--------+------+---------+----------+----------+



In [0]:
# STEP 3: Adding hash key to both the dfs

src_customers_df = src_customers_df.withColumn("src_hash_key", xxhash64("name", "city", "salary"))
src_customers_df.show()

tgt_customers_df = tgt_customers_df.withColumn("tgt_hash_key", xxhash64("name", "city", "salary"))
tgt_customers_df.show()

+-----------+---------+--------+------+--------------------+
|customer_id|     name|    city|salary|        src_hash_key|
+-----------+---------+--------+------+--------------------+
|          1|    Ayush|   Noida| 50000|-2909879855720589612|
|          5| Pragnesh|Gr.Noida| 34000| -257979407294553456|
|          6|Priyanshu|   Khora| 34000|-2920003710670612202|
+-----------+---------+--------+------+--------------------+

+-----------+--------+--------+------+---------+----------+----------+--------------------+
|customer_id|    name|    city|salary|is_active|start_date|  end_date|        tgt_hash_key|
+-----------+--------+--------+------+---------+----------+----------+--------------------+
|          1|   Ayush|   Noida| 50000|        Y|2026-07-19|9999-12-31|-2909879855720589612|
|          3|  Gourav|  Colony| 34000|        Y|2026-07-19|9999-12-31|-4066889726975237581|
|          5|Pragnesh|Gr.Noida| 33000|        Y|2026-07-19|9999-12-31| -647272837507553151|
|          2|  Piyus

In [0]:
# STEP 4: Rename source and target columns to match the target table

src_customers_df = src_customers_df.withColumnRenamed("customer_id", "src_customer_id").withColumnRenamed("name", "src_name").withColumnRenamed("city", "src_city").withColumnRenamed("salary", "src_salary")
src_customers_df.show()

tgt_customers_df = tgt_customers_df.withColumnRenamed("customer_id", "tgt_customer_id").withColumnRenamed("name", "tgt_name").withColumnRenamed("city", "tgt_city").withColumnRenamed("salary", "tgt_salary")
tgt_customers_df.show()

+---------------+---------+--------+----------+--------------------+
|src_customer_id| src_name|src_city|src_salary|        src_hash_key|
+---------------+---------+--------+----------+--------------------+
|              1|    Ayush|   Noida|     50000|-2909879855720589612|
|              5| Pragnesh|Gr.Noida|     34000| -257979407294553456|
|              6|Priyanshu|   Khora|     34000|-2920003710670612202|
+---------------+---------+--------+----------+--------------------+

+---------------+--------+--------+----------+---------+----------+----------+--------------------+
|tgt_customer_id|tgt_name|tgt_city|tgt_salary|is_active|start_date|  end_date|        tgt_hash_key|
+---------------+--------+--------+----------+---------+----------+----------+--------------------+
|              1|   Ayush|   Noida|     50000|        Y|2026-07-19|9999-12-31|-2909879855720589612|
|              3|  Gourav|  Colony|     34000|        Y|2026-07-19|9999-12-31|-4066889726975237581|
|              5

In [0]:
# STEP 5: Join the source and target

joined_df = src_customers_df.alias("src").join(other=tgt_customers_df.alias("tgt"), on=src_customers_df.src_customer_id == tgt_customers_df.tgt_customer_id, how="left")

joined_df.display()

src_customer_id,src_name,src_city,src_salary,src_hash_key,tgt_customer_id,tgt_name,tgt_city,tgt_salary,is_active,start_date,end_date,tgt_hash_key
1,Ayush,Noida,50000,-2909879855720589612,1,Ayush,Noida,50000,Y,2026-07-19,9999-12-31,-2909879855720589612
5,Pragnesh,Gr.Noida,34000,-257979407294553456,5,Pragnesh,Gr.Noida,33000,Y,2026-07-19,9999-12-31,-647272837507553151
6,Priyanshu,Khora,34000,-2920003710670612202,null,null,null,null,null,null,null,null


In [0]:
# STEP 6: Finding new and updated records

required_records_df = (
joined_df.filter("src_hash_key != tgt_hash_key or tgt_customer_id IS NULL")
.select("src.*")
.withColumnsRenamed({"src_customer_id": "customer_id", "src_name": "name", "src_city": "city", "src_salary": "salary"})
.drop("src_hash_key")
.withColumn("is_active", lit("Y"))
.withColumn("start_date", lit(current_timestamp()))
.withColumn("end_date", lit("9999-12-31").cast("date"))
)

required_records_df.display()

customer_id,name,city,salary,is_active,start_date,end_date
5,Pragnesh,Gr.Noida,34000,Y,2026-07-19T14:07:34.248Z,9999-12-31
6,Priyanshu,Khora,34000,Y,2026-07-19T14:07:34.248Z,9999-12-31


In [0]:
# create delta table
tgt_customers_dlt = DeltaTable.forName(spark, "mydb.CUST_DIM")

In [0]:
# STEP 7: End the old records and insert the new records using MERGE
(tgt_customers_dlt.alias("tgt").merge(
    source=required_records_df.alias("src"),
    condition="tgt.customer_id = src.customer_id"
)
.whenMatchedUpdate(
    set={
        "is_active": lit("N"),
        "end_date": current_date() - 1
    }
)
.whenNotMatchedInsertAll()
.execute())


DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
# STEP 8: Insert the updated records using MERGE
(tgt_customers_dlt.alias("tgt").merge(
    source=required_records_df.alias("src"),
    condition="tgt.customer_id = src.customer_id AND tgt.is_active = 'Y'"
)
.whenNotMatchedInsertAll()
.execute())

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql 
select * from mydb.cust_dim;

customer_id,name,city,salary,is_active,start_date,end_date
5,Pragnesh,Gr.Noida,33000,N,2026-07-19,2026-07-18
6,Priyanshu,Khora,34000,Y,2026-07-19,9999-12-31
1,Ayush,Noida,50000,Y,2026-07-19,9999-12-31
3,Gourav,Colony,34000,Y,2026-07-19,9999-12-31
1,Ayush,Noida,30000,N,2026-07-19,2026-07-18
3,Gaurav,Colony,34000,N,2026-07-19,2026-07-18
5,Pragnesh,Gr.Noida,34000,Y,2026-07-19,9999-12-31
2,Piyush,Sorkha,20000,Y,2026-07-19,9999-12-31
4,Bhonu,Sorkha,20000,Y,2026-07-19,9999-12-31
